# Project- Marketing Brochure:
Build a tool that infers important and relevant information for a product, creates a marketing brochure for the product, and translates the brochure to another language (based on user input).
### Activate your environment as Kernel

### Setup Environment

In [ ]:
!conda init
!conda activate my_llms
!python -m ipykernel install --user --name=my_llms --display-name="Python (my_llms)"

### Imports

In [ ]:
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
import json
from openai import OpenAI
import os
import requests
import ollama
import signal
import gradio as gr
from inputimeout import inputimeout, TimeoutOccurred

### Timeout function

In [2]:
# define function for timeout for user input

def timeout_handler(signum, frame):
    raise TimeoutError

# Set the signal handler and a 5-second alarm
signal.signal(signal.SIGALRM, timeout_handler)
signal.alarm(5)

0

In [ ]:
!conda list ollama
!ollama pull llama3.2

### Constants

In [ ]:
OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

### Environment setup 2

In [6]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print('API key looks fine')
else:
    print('Somehting is wrong with the API key.')

openai = OpenAI()

API key looks fine


### A class to represent a Webpage

Some websites need you to use proper headers when fetching them.

In [ ]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent contents of a Website
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [ ]:
w = Website("https://www.apple.com/apple-watch-series-10/")
w.text          # displays all the content of the webpage. Since it is a very large piece of text, it is not pushed to the repo.

### A function to use the Website class

In [9]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    return result

### User prompt

In [10]:
def get_brochure_user_prompt(product_name, url):
    user_prompt = f"You are looking at a product called: {product_name}\n"
    user_prompt += f"Here are the contents of its landing page; use this information to build a short brochure of the product in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

### System prompt

In [ ]:
system_prompt = """
You are provided with product information found on a webpage. \
You are able to decide which of the text would be most relevant to include in a marketing brochure about the product for prospective customers, \
such as specifications, uniqueness, price, etc. \
Respond in Markdown.
"""

## With GPT (Markdown)

### A funtion for translating brochure to another language

In [ ]:
def translate_brochure(info, language, model):
    print("#" * 50)
    print("#" * 20, f"{language} Version",  "#" * 20)
    system_prompt2 = f"""
    Translate the following marketing brochure into {language} language.
    Respond in Markdown.
    """
    messages = [
                {"role": "system", "content": system_prompt2},
                {"role": "user", "content": info}
            ]
    if model==1:
        response = openai.chat.completions.create(
            model=MODEL_GPT,
            messages = messages,
        )
        result = response.choices[0].message.content
    else:
        response = ollama.chat(model=MODEL_LLAMA, messages=messages)
        result = response['message']['content']
    return result

# A function for creating brochure

In [ ]:
# Parameters:
#   product_name: Name of the product
#   url: website url of the product
#   model: GPT4 or Ollama; 1 for GPT and 0 for Ollama

def create_brochure(product_name, url, model):
    try:
        language = input("Enter language for translation (leave blank if no translation): ")
    except TimeoutError:
        language=''
    finally:
        signal.alarm(0) 
    messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": get_brochure_user_prompt(product_name, url)}
            ]
    if model==1:
        response = openai.chat.completions.create(
            model=MODEL_GPT,
            messages=messages,
        )
        result = response.choices[0].message.content
    elif model==0:
        response = ollama.chat(model=MODEL_LLAMA, messages=messages)
        result = response['message']['content']
    else:
        raise ValueError("Unknown model. Please choose 1 for GPT and 0 for Ollama")


    display(Markdown(result))
    if language:
        translate = translate_brochure(result, language, model)
        display(Markdown(translate))

In [16]:
create_brochure("Apple Watch Series 10", "https://www.apple.com/apple-watch-series-10/",0)

**Apple Watch Series 10 Brochure**
=====================================

**Overview**
------------

Introducing the Apple Watch Series 10, the thinnest and most advanced watch yet from Apple. With a massive display and cutting-edge technology, this watch is designed to inspire you to stay active and healthy.

**Tech Specs**
-------------

* **Largest Screen Area**: Series 10 boasts the largest screen area of any Apple Watch, with up to 75% more screen real estate than previous models.
* **Fastest-Charging Battery**: Get 80% battery life in just 30 minutes with our fastest-charging technology yet.
* **Advanced Health Insights**: Track your sleep, heart health, and other vital metrics with our advanced sensors.

**Key Features**
----------------

* **Jet Black Aluminum Case**: A sleek and sophisticated design that's perfect for those who want a timeless look.
* **Titanium Finishes**: Choose from three polished titanium finishes that are as beautiful as they are durable.
* **ECG App**: Get an ECG reading similar to a single-lead electrocardiogram with the built-in ECG app.
* **Cycle Tracking**: Track your temperature while you sleep and get a retrospective estimate of when you likely ovulated.

**Fitness and Activity**
-----------------------

* **Motivates You to Stay Active**: Apple Watch Series 10 encourages you to close your Move, Exercise, and Stand rings every day.
* **Water Temperature Sensor**: Get more information for your swim workouts with our water temperature sensor.
* **Depth Sensor**: Make Series 10 great for swimming and snorkeling with our depth sensor.

**Get Started**
----------------

Order now and get 3 months of Fitness+ free when you buy any Apple Watch. Visit our website to learn more and purchase today!

##################################################
#################### german Version ####################


**Apple Watch Series 10 - Deutsche Broschure**
=============================================

**Übersicht**
-------------

Wir freuen uns, die neue Apple Watch Serie 10 vorzustellen, die schmalste und fortschrittlichste Uhr von Apple. Mit einem großen Display und der neuesten Technologie ist diese Uhr darauf ausgelegt, Sie zu inspirieren, aktiv und gesund zu sein.

**Technische Daten**
-----------------

* **Größtes Bildfeld**: Die Serie 10 verfügt über das größte Bildfeld aller Apple Watches, mit bis zu 75 % mehr Bildfläche als frühere Modelle.
* **Schnellste Aufladung**: Erhalten Sie 80 % der Batterieleistung in nur 30 Minuten mit unserer schnellsten Aufladungs-Technologie noch.
* **Fortgeschrittene Gesundheitsinsichten**: Verfolgen Sie Ihre Schlaf, Herzgesundheit und andere wertvolle Metriken mit unseren fortgeschrittenen Sensoren.

**Wichtige Funktionen**
----------------------

* **Jet Black Aluminiumgehäuse**: Eine elegante und zeitlose Design-Option für jene, die ein unvergängliches Aussehen bevorzugen.
* **Titanium-Fertigungen**: Wählen Sie aus drei hochpolierten Titanium-Fertigungen, die genauso schön wie robust sind.
* **ECG-Anwendung**: Erhalten Sie eine ECG-Vorlesung, ähnlich einer Ein-Leit-Elektrokardiogramm-Anwendung mit der integrierten ECG-Anwendung.
* **Schwangerschaftsverlauf**: Verfolgen Sie Ihre Körpertemperatur während des Schlafes und erhalten Sie ein rückblickendes Schätzungsziel für den Zeitpunkt, zu dem Sie wahrscheinlich schwanger waren.

**Fitnes und Aktivität**
-------------------------

* **Motiviert Sie zur regelmäßigen körperlichen Betätigung**: Die Apple Watch Serie 10 ermutigt Sie, die Move-, Exercise- und Stand-Bewegungsringe jeden Tag zu schließen.
* **Wasser-Temperatur-Sensor**: Erhalten Sie mehr Informationen für Ihre Schwimmsport-Arbeiten mit unserem Wasser-Temperatur-Sensor.
* **Tiefensensor**: Machen Sie Serie 10 großartig für Schwimmen und Schnorcheln mit unseren Tiefensensoren.

**Mit Beginn**
----------------

Bestellen Sie jetzt und genießen Sie drei Monate von Fitness+ kostenlos, wenn Sie eine Apple Watch kaufen. Besuchen Sie unsere Website, um mehr zu erfahren und heute kaufen!

## Streaming brochure

In [14]:
def stream_brochure(product_name, url, model, language=None): 
    messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": get_brochure_user_prompt(product_name, url)}
            ]
    result=''
    if model=='GPT':
        stream = openai.chat.completions.create(
            model=MODEL_GPT,
            messages=messages,
            stream=True
        )
        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                result += delta
                yield result
        
    elif model=='Ollama':
        stream = ollama.chat(model=MODEL_LLAMA, messages=messages, stream=True)
        for chunk in stream:
            delta = chunk['message']['content']
            result += delta
            yield result
    else:
        raise ValueError("Unknown model. Please choose 1 for GPT and 0 for Ollama")
    
    #if language and language!='None':
    #    translate = translate_brochure(result, language, model)
    #return result, translate


## Visualize UI with Gradio

In [20]:
view = gr.Interface(
    fn = stream_brochure,
    inputs=[gr.Textbox(label="Product Name:"), 
            gr.Textbox(label="Product url:"),
            gr.Dropdown(["GPT", "Ollama"], label='Select Model', value="GPT"),
            #gr.Dropdown([None, "German", "French", "Hindi", "Spanish", "Dutch"], label='Select Language', value=None)
            ],
    outputs=[gr.Markdown(label="Original Brochure:")],
    flagging_mode="never",
)

view.launch()



* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.
